In [1]:
import numpy as np

def lagrange(x_nos, y_nos, x):
    n = len(x_nos)
    p = 0.0
    for i in range(n):
        L = 1.0
        for j in range(n):
            if j != i:
                L *= (x - x_nos[j]) / (x_nos[i] - x_nos[j])
        p += y_nos[i] * L
    return p

def bisseccao(g, a, b, tol=1e-6, max_iter=10000):
    fa = g(a)
    for _ in range(max_iter):
        m = (a + b) / 2
        fm = g(m)
        if abs(fm) < tol or (b - a) / 2 < tol:
            return m
        if fa * fm < 0:
            b = m
        else:
            a = m
            fa = fm
    return (a + b) / 2

# Parâmetros
Is_d      = 1e-12   # corrente de saturação (A)
nu        = 1       # fator de idealidade
VT        = 25e-3   # tensão térmica (V)
R         = 1e3     # resistência (Ω)
Vs        = 5.0     # tensão da fonte (V)
tolerance = 1e-6

def f(Vd):
    return (Vs - Vd) / R - Is_d * (np.exp(Vd / (nu * VT)) - 1)

x_nos = np.linspace(0.50, 0.60, 5)
y_nos = f(x_nos)

In [2]:
Vd_lagrange = bisseccao(lambda x: lagrange(x_nos, y_nos, x), 0.50, 0.60)
I_lagrange  = Is_d * (np.exp(Vd_lagrange / (nu * VT)) - 1)

print("=== Lagrange ===")
print(f"Vd = {Vd_lagrange:.6f} V")
print(f"I  = {I_lagrange*1e3:.4f} mA")

=== Lagrange ===
Vd = 0.555536 V
I  = 4.4735 mA


In [3]:
def newton_coef(x, y):
    n = len(x)
    c = y.copy().astype(float)
    for j in range(1, n):
        for i in range(n - 1, j - 1, -1):
            c[i] = (c[i] - c[i - 1]) / (x[i] - x[i - j])
    return c

def newton_poly(x_nos, c, x):
    n = len(x_nos)
    p = c[n - 1]
    for i in range(n - 2, -1, -1):
        p = p * (x - x_nos[i]) + c[i]
    return p

coef = newton_coef(x_nos, y_nos)

print("Coeficientes das diferenças divididas:")
for k, ck in enumerate(coef):
    print(f"  c[{k}] = {ck:.6e}")

Vd_newton = bisseccao(lambda x: newton_poly(x_nos, coef, x), 0.50, 0.60)
I_newton  = Is_d * (np.exp(Vd_newton / (nu * VT)) - 1)

print("\n=== Newton ===")
print(f"Vd = {Vd_newton:.6f} V")
print(f"I  = {I_newton*1e3:.4f} mA")

Coeficientes das diferenças divididas:
  c[0] = 4.014835e-03
  c[1] = -3.434602e-02
  c[2] = -1.145957e+00
  c[3] = -2.625437e+01
  c[4] = -4.511240e+02

=== Newton ===
Vd = 0.555536 V
I  = 4.4735 mA
